## xtest code vulnerable users

In [1]:
!pip install pandas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print(pd.__version__)

3.0.2


In [2]:
df = pd.read_csv("xtest.csv")
df.head()

,testid,NON_MTRST_TYPE_CL,MOST_HRMFL_EVT_CL
0,5059685,VU1: Pedestrian,V1:(Collision with pedestrian)
1,5053438,VU2: Bicyclist,"V1:(Collision with cyclist (bicycle, tricycle,..."
2,5059947,VU1: Pedestrian,V1:(Collision with pedestrian) / V2:(Collision...
3,5061858,VU1: Pedestrian,V1:(Collision with pedestrian)
4,5112584,VU2: Other,V1:(Collision with tree)


In [3]:
df.info(show_counts=True, memory_usage=True, verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   testid             159 non-null    int64
 1   NON_MTRST_TYPE_CL  159 non-null    str  
 2   MOST_HRMFL_EVT_CL  156 non-null    str  
dtypes: int64(1), str(2)
memory usage: 3.9 KB


In [4]:
df.shape #number of rows and columns

(159, 3)

In [5]:
# The following code creates new columns with numerical values:
# PEDESTRIAN column = 1 if a pedestrian or similar low-speed user was involved in a crash, otherwise = 0
# CYCLIST column = 1 if a cyclist or similar micromobility user was involved in a crash, otherwise = 0
# OTHER column = 1 if higher-speed user or other type listed above or unknown was involved in a crash, otherwise = 0

In [6]:
col1 = df["NON_MTRST_TYPE_CL"]
col2 = df["MOST_HRMFL_EVT_CL"]

# uses np from numpy library above
df["PEDESTRIAN"] = np.where(
    col1.str.contains("Pedestrian|Electric Personal Assistive Mobility Device|Wheelchair|Responder|Worker", case=False, na=False) |
    col2.str.contains("Pedestrian", case=False, na=False),
    1,
    0
)

#contains Bicyclist in column1 or Cyclist in column2 = 1, but if "Motorized" = false
df["CYCLIST"] = np.where(
    (
        col1.str.contains("Bicyclist|Cyclist|Skater|Non-Motorized Scooter Rider|Micromobility|Skateboarder|Tricyclist", case=False, na=False) |
        col2.str.contains("Cyclist", case=False, na=False)
    )
    &
    ~(
        col1.str.contains("Motorized", case=False, na=False) |
        col2.str.contains("Motorized", case=False, na=False)
    ),
    1,
    0
)

#contains Motorized Bicyclist or Motorized Scooter or moped or Other types below
df["OTHER"] = np.where(
    (
        col1.str.contains("Other|Motorized Bicyclist|Motorized Scooter Rider|Passenger|Farm|Unknown", case=False, na=False) |
        col2.str.contains("Other Vulnerable|moped", case=False, na=False)
    ),
    1,
    0
)
df.head()

,testid,NON_MTRST_TYPE_CL,MOST_HRMFL_EVT_CL,PEDESTRIAN,CYCLIST,OTHER
0,5059685,VU1: Pedestrian,V1:(Collision with pedestrian),1,0,0
1,5053438,VU2: Bicyclist,"V1:(Collision with cyclist (bicycle, tricycle,...",0,1,0
2,5059947,VU1: Pedestrian,V1:(Collision with pedestrian) / V2:(Collision...,1,0,0
3,5061858,VU1: Pedestrian,V1:(Collision with pedestrian),1,0,0
4,5112584,VU2: Other,V1:(Collision with tree),0,0,1


In [7]:
df.to_csv("xtest-output.csv")